In [2]:
import numpy as np
import gfapy
import networkx as nx
import gurobipy as gp
from gurobipy import GRB

In [3]:
filename = 'test_N3_W4'
filepath = f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/{filename}.gfa'

gfa = gfapy.Gfa.from_file(filepath, vlevel=0)


# TODO: use copy numbers from pathfinder
graph = nx.DiGraph()
for index, segment_line in enumerate(gfa.segments):
    graph.add_node(f'{segment_line.name}_+', weight=segment_line.SC, start=segment_line.st)
    graph.add_node(f'{segment_line.name}_-', weight=segment_line.SC, start=segment_line.st)
for edge_line in gfa.edges:
    v1 = edge_line.sid1
    v2 = edge_line.sid2
    graph.add_edges_from([
        (f'{v1.name}_{v1.orient}', f'{v2.name}_{v2.orient}'),
    ])
    v1.invert()
    v2.invert()
    graph.add_edges_from([
        (f'{v2.name}_{v2.orient}', f'{v1.name}_{v1.orient}'),
    ])

nodes = list(graph.nodes)
N = len(nodes)
n = int(np.ceil(np.log2(N+1)))
total_weight = int(sum(graph.nodes[node]["weight"] for node in nodes) / 2)
T = int(1.1 * total_weight)

In [ ]:


with gp.Env() as env, gp.Model(env=env) as model:
    model_x_vars = model.addMVar(shape=(T,n), vtype=GRB.BINARY, name="x")
    model_X_vars = model.addMVar(shape=T, lb=0, ub=N, vtype=GRB.INTEGER, name="X")
    model.setObjective(
        model_x_vars[0]**2, 
        GRB.MINIMIZE
        )
    model.Params.BestObjStop = 0
    for t in range(T):
        model.addConstr(
            model_X_vars[t] == sum([model_x_vars[t][k] * 2 ** k for k in range(n)]) 
        )
    model.optimize()